# Named Entity Recognition — Azure AI Language

This notebook uses **Azure AI Language** to identify and categorize *named entities* in text — people, organizations, locations, dates, quantities, and more.

NER is useful for information extraction, knowledge graph population, and building search indexes.

In [ ]:
%pip install azure-ai-textanalytics azure-core python-dotenv --quiet

In [ ]:
import os
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["FOUNDRY_AI_SERVICES_ENDPOINT"]
api_key = os.environ["FOUNDRY_AI_SERVICES_KEY"]

client = TextAnalyticsClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
print("Language client ready.")

In [ ]:
# Sample documents for NER
documents = [
    "Satya Nadella is the CEO of Microsoft, headquartered in Redmond, Washington.",
    "The Eiffel Tower was built in 1889 and stands 330 meters tall in Paris, France.",
    "On January 15, 2024, OpenAI announced a new partnership worth $10 billion with Microsoft Azure.",
]

print(f"Recognizing entities in {len(documents)} documents...")

In [ ]:
# Perform Named Entity Recognition
results = client.recognize_entities(documents=documents)

for i, result in enumerate(results):
    if result.is_error:
        print(f"Document {i + 1} error: {result.error.code} - {result.error.message}")
        continue

    print(f"\n--- Document {i + 1} ---")
    print(f"Text: {documents[i]}")
    print("Entities:")
    for entity in result.entities:
        print(f"  '{entity.text}' → category: {entity.category}, "
              f"subcategory: {entity.subcategory or '—'}, "
              f"confidence: {entity.confidence_score:.4f}")

In [ ]:
# Group entities by category across all documents
from collections import defaultdict

by_category = defaultdict(list)

results2 = client.recognize_entities(documents=documents)
for result in results2:
    if not result.is_error:
        for entity in result.entities:
            by_category[entity.category].append(entity.text)

print("\nEntities grouped by category:")
for category, entities in sorted(by_category.items()):
    unique_entities = sorted(set(entities))
    print(f"  {category}: {', '.join(unique_entities)}")